# HydroOpsLM: understand the LLM decision path / 理解 LLM 決策路徑

## Goal / 目標
HydroOpsLM studies LLM reservoir operations through the HydroCNHS Dam API. Follow info + independent prompt + optional context → LLM → reader → proposal → checker → execution → feedback. Audit evaluates configured rules; trace records the chain. Info changes daily; templates, model settings and audit rubrics stay fixed for one run. Only final messages are visible to the model. Requested release is a proposal; common physics determines implemented release, spill and storage.

Read [LLM modules](../docs/LLM_MODULES.md) and [generic settings](../docs/USER_GUIDE.md).

![Architecture](../docs/figures/hydrocnhs-coupling-framework-en.png)

HydroOpsLM 透過 HydroCNHS Dam API 研究 LLM 水庫操作。流程為 info＋獨立 prompt＋選用 context → LLM → reader → proposal → checker → execution → feedback。Audit 評估指定規則，trace 保存鏈條。Info 逐日更新；同一 run 的模板、模型及 audit 規則固定。模型只看見最終 messages。請求放流是 proposal，實際放流、spill 與蓄水由共同物理計算。詳見正式模組及設定手冊。

## Setup / 準備
Python 3.12 with repository dependencies. Local files only; no provider calls. / 使用 Python 3.12 與既有依賴；使用局部檔案，不呼叫 provider。

In [1]:
from pathlib import Path
import json, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/hydrocnhs').is_dir())
sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT))
from llm_reservoir_agent.workflow import load_settings, run_settings, verify_settings_output
from llm_reservoir_agent.verification import verify_run
from unittest.mock import patch
from tutorials.inspect_decision_flow import show_decision_flow
def ensure_offline(config):
    settings = load_settings(ROOT/config)
    assert settings.report['planned_llm_calls'] == 0
    output = Path(settings.report['output'])
    if not output.exists():
        with patch('urllib.request.urlopen', side_effect=AssertionError('network forbidden')):
            run_settings(settings)
    return settings, verify_settings_output(settings)
print('Checkout:', ROOT.name)


Checkout: HydroCNHS-Delaware-Operations


## Steps

### Inspect bundled capabilities / 查看保留的原生能力

In [2]:
import hydrocnhs
source=json.loads((ROOT/'third_party/hydrocnhs/SOURCE.json').read_text())
print('Native HydroCNHS commit:', source['commit'])
print('Rainfall-runoff:', sorted(p.name for p in (ROOT/'src/hydrocnhs/rainfall_runoff_model').glob('*.py')))
print('Methods: rule-based benchmark, zero-shot, DKP')

Native HydroCNHS commit: fdd5097a3fcabb312c6c3739efc71da2d67c47da
Rainfall-runoff: ['__init__.py', 'abcd.py', 'gwlf.py', 'pet_hamon.py']
Methods: rule-based benchmark, zero-shot, DKP


## Checks / 檢查
Review printed inputs, counts, sources and residuals. Synthetic replies validate interfaces only. / 核對輸入、計數、來源與殘差；合成回覆僅驗證介面。

## Next steps / 下一步
Use lesson 01 to distinguish already routed inflow from meteorological forcing. Calibration/validation remain native HydroCNHS workflows; no recalibration is performed here.

下一課區分已 routing 入流與氣象 forcing。C&V 保留原生 HydroCNHS 工作流程；此處不重新校正。